# CPU Checkpoint and Restore (Pause and Resume)

Pausing a Workspace checkpoints the pod to Google Cloud Storage (`SNAPSHOT_GCS_BUCKET`); resuming restores it.
This notebook verifies that **stateful pause and resume works for CPU workloads**, preserving:

- **Live Python kernel execution state** and in-memory variables without reloading.
- **Bit-exact in-memory datasets**: multi-hundred-megabyte data structures sitting in RAM come back intact.
- **Process continuity**: the process ID (PID) matches, ruling out a container reboot or fresh restart.
- **Thread state & frozen time**: background threads freeze during the pause and resume counting afterward.
- **Live compute readiness**: CPU operations immediately succeed after restore without re-initialization.

Unlike a stateless pause (which terminates the pod and discards non-disk memory), snapshot-enabled pause captures
the full container memory and process tree using GKE Pod Snapshots under the gVisor sandbox.

**Requirements:** Standard CPU image (`jupyterlab-cpu` or `jupyter-scipy`) with a CPU pod config
(e.g., `Small CPU` or `Medium CPU`) on a snapshot-enabled `WorkspaceKind` (`jupyterlab-resumable`)
or any `Workspace` annotated with `podsnapshot.gke.kubeflow.org/enabled: "true"`.

**How to run:** Run steps 1–2, pause the Workspace, resume it, then run **only** step 3.
Re-running steps 1–2 after the resume would overwrite the state and defeat the test.

In [ ]:
# Step 1 — confirm CPU environment and system specs.
import os
import platform
import socket
import sys
import time

print(f"host        : {socket.gethostname()}")
print(f"pid         : {os.getpid()}")
print(f"python      : {sys.version.split()[0]}")
print(f"platform    : {platform.platform()}")

try:
    with open("/proc/meminfo", "r") as f:
        for line in f:
            if line.startswith(("MemTotal:", "MemAvailable:")):
                print(f"{line.split(':')[0].strip().lower():12}: {line.split(':')[1].strip()}")
except Exception:
    pass

print("\nCPU environment verified. Ready to allocate in-memory state.")

In [ ]:
# Step 2 — create in-memory state, compute its exact fingerprint, and start background tick tracker.
import hashlib
import threading
import numpy as np

PID = os.getpid()
CREATED_AT = time.time()

# Create an in-memory dataset (~160 MB of float64 data)
print("Generating in-memory data array (~160 MB)...", end=" ", flush=True)
t0 = time.time()
# Deterministic sequence: 20 million 64-bit floats
N = 20_000_000
DATA = np.linspace(0.0, 1000.0, num=N, dtype=np.float64)
# Apply a non-trivial transformation
DATA = np.sin(DATA) * np.cos(DATA * 0.5)
GENERATE_SECONDS = time.time() - t0
print(f"done in {GENERATE_SECONDS:.2f}s")

# Bit-exact SHA-256 fingerprint over the raw memory buffer
DATA_FP = hashlib.sha256(DATA.tobytes()).hexdigest()
DATA_SUM = float(DATA.sum())
DATA_BYTES = DATA.nbytes
SECRET_TOKEN = f"stateful-cpu-token-{PID}-{int(CREATED_AT)}"

# Background thread that ticks every second while the process runs.
# When paused via PodSnapshot, process execution is frozen and ticks stop advancing.
TICKS = 0

def _tick():
    global TICKS
    while True:
        time.sleep(1)
        TICKS += 1

t = threading.Thread(target=_tick, daemon=True)
t.start()

print(f"pid          : {PID}")
print(f"elements     : {N:,}")
print(f"memory size  : {DATA_BYTES / (1024**2):.1f} MiB")
print(f"data sum     : {DATA_SUM:.6f}")
print(f"data hash    : {DATA_FP[:16]}...{DATA_FP[-8:]}")
print(f"token string : {SECRET_TOKEN}")
print("\nIn-memory state established! Now proceed to pause the Workspace.")

## Now pause the Workspace

The data and background thread are now resident in memory. Note the `pid` and `data hash` printed above, then pause the workspace:

1. In the Kubeflow Workspaces UI, click **Stop** (or **Pause**) on this Workspace.
2. Wait for the workspace status in the UI to transition to **Paused**.
3. Click **Start** (or **Resume**) to restore the workspace.

Once the workspace returns to `Running (1/1)`, reopen this notebook, and run **Step 3 below without re-running Steps 1 and 2**.

In [ ]:
# Step 3 — run this AFTER the resume. Do NOT re-run Steps 1 or 2!
try:
    PID, CREATED_AT, DATA, DATA_FP, DATA_SUM, SECRET_TOKEN, TICKS
except NameError:
    raise RuntimeError(
        "The globals from step 2 are gone! The pod was restarted or recreated from scratch "
        "rather than restored from a GKE PodSnapshot. Checkpoint/restore FAILED."
    ) from None

checks = []

# 1. Verify process continuity
current_pid = os.getpid()
checks.append(("same process", current_pid == PID, f"pid {current_pid} matches original {PID}"))

# 2. Verify bit-exact in-memory array integrity
current_fp = hashlib.sha256(DATA.tobytes()).hexdigest()
checks.append((
    "array bit-identical",
    current_fp == DATA_FP,
    f"{current_fp[:16]}... matches original {DATA_FP[:16]}...",
))

# 3. Verify numerical consistency
current_sum = float(DATA.sum())
sum_diff = abs(current_sum - DATA_SUM)
checks.append((
    "numerical consistency",
    sum_diff < 1e-12,
    f"sum diff = {sum_diff:.2e}",
))

# 4. Verify in-memory string token
expected_token = f"stateful-cpu-token-{PID}-{int(CREATED_AT)}"
checks.append((
    "in-memory variables preserved",
    SECRET_TOKEN == expected_token,
    "string secret intact",
))

# 5. Verify process was frozen while paused
elapsed_wallclock = time.time() - CREATED_AT
frozen_seconds = elapsed_wallclock - TICKS
checks.append((
    "process was frozen during pause",
    frozen_seconds > 5,
    f"~{frozen_seconds:.1f}s paused, {TICKS}s running (wallclock: {elapsed_wallclock:.1f}s)",
))

# 6. Verify CPU compute continues to work seamlessly on restored state
try:
    t_compute = time.time()
    new_result = float(np.mean(DATA[:1_000_000] ** 2))
    compute_time = time.time() - t_compute
    compute_ok = True
    compute_detail = f"np.mean(x^2) = {new_result:.6f} in {compute_time * 1000:.1f}ms"
except Exception as exc:
    compute_ok = False
    compute_detail = f"{type(exc).__name__}: {exc}"
checks.append(("cpu compute operational", compute_ok, compute_detail))

# Report results
label_width = max(len(name) for name, _, _ in checks)
print("=" * 75)
print("CPU CHECKPOINT & RESTORE VERIFICATION REPORT")
print("=" * 75)
for name, ok, detail in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {name.ljust(label_width)}  |  {detail}")
print("=" * 75)

passed = all(ok for _, ok, _ in checks)
print(f"Final Result: CPU checkpoint/restore {'PASSED' if passed else 'FAILED'}")
if passed:
    print("\nAll in-memory state, live variables, background threads, and compute resumed seamlessly!")

## Reading the Result

| Check | Expected | Failure Meaning |
|---|---|---|
| `NameError` on Step 3 | Globals present | Pod was recreated without restore (check `podsnapshot.gke.kubeflow.org/enabled`). |
| `same process` | PID matches | Process restarted rather than restored. |
| `array bit-identical` | Hash matches | Memory corruption during checkpoint or restore. |
| `numerical consistency` | Difference < 1e-12 | Arithmetic drift or memory truncation. |
| `process was frozen` | Frozen > 5s | Workspace was not paused or resume happened immediately without snapshot. |
| `cpu compute operational` | Success | Kernel execution pipeline restored and responsive. |

### Typical Performance on CPU (e.g. n1-standard-4 / e2-standard-4)

| Operation | Duration | Notes |
|---|---|---|
| **Checkpoint (pause)** | **12–25 s** | ~500 MB – 1.2 GB snapshot written to GCS (`SNAPSHOT_GCS_BUCKET`) |
| **Restore (resume)** | **4–8 s** | Streamed directly from GCS into container memory |

To monitor the snapshot controller and GKE PodSnapshot resources from a cluster terminal:

```bash
# Check active podsnapshots and storage locations
kubectl -n <namespace> get podsnapshots

# Inspect snapshot controller logs
kubectl -n kubeflow-workspaces logs -l app=gke-workspace-snapshot-addon -f --prefix
```